<a href="https://colab.research.google.com/github/HectorChona/telecom-analysis./blob/main/S7_Version_Estudiante_Project_ConnectaTel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis ConnectaTel

Como **analista de datos**, tu objetivo es evaluar el **comportamiento de los clientes** de una empresa de telecomunicaciones en Latinoamérica, ConnectaTel.

Trabajaremos con información registrada **hasta el año 2024**, lo cual permitirá analizar el comportamiento del negocio dentro de ese periodo.

Para ello trabajarás con tres datasets:  

- **plans.csv** → información de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)  
- **users.csv** → información de los clientes (edad, ciudad, fecha de registro, plan, churn)  
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)  

Deberás **explorar**, **limpiar** y **analizar** estos datos para construir un **perfil estadístico** de los clientes, detectar **comportamientos atípicos** y crear **segmentos de clientes**.  

Este análisis te permitirá **identificar patrones de consumo**, **diseñar estrategias de retención** y **sugerir mejoras en los planes** ofrecidos por la empresa.

> 💡 Antes de empezar, recuerda pensar de forma **programática**: ¿qué pasos necesitas? ¿En qué orden? ¿Qué quieres medir y por qué?


---
## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarte con la estructura de los tres datasets**.  
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectarás posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**  
- Importa las librerías necesarias (por ejemplo `pandas`, `seaborn`, `matplotlib.pyplot`)
- Carga los archivos CSV usando `pd.read_csv()`:
  - **`/datasets/plans.csv`**  
  - **`/datasets/users_latam.csv`**  
  - **`/datasets/usage.csv`**  
- Guarda los DataFrames en las variables: `plans`, `users`, `usage`.  
- Muestra las primeras filas de cada DataFrame usando `.head()`.


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
# cargar archivos
plans = pd.read_csv('datasets/plans.csv')
users = pd.read_csv('datasets/users_latam.csv')
usage = pd.read_csv('datasets/usage.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/plans.csv'

In [ ]:
print('--- PLANS ---')
display(plans.head(5))

In [ ]:
print('\n--- USERS ---')
display(users.head())

In [ ]:
print('\n--- USAGE ---')
display(usage.head())

**Tip:** Si no usas `print()` la tabla se vera mejor.

### 1.2 Exploración de la estructura de los datasets

**🎯 Objetivo:**  
Conocer la **estructura de cada dataset**, revisar cuántas filas y columnas tienen, identificar los **tipos de datos** de cada columna y detectar posibles **inconsistencias o valores nulos** antes de iniciar el análisis.

**Instrucciones:**  
- Revisa el **número de filas y columnas** de cada dataset usando `.shape`.  
- Usa `.info()` en cada DataFrame para obtener un **resumen completo** de columnas, tipos de datos y valores no nulos.  

In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

In [ ]:
plans.info()

In [ ]:
users.info() # inspección de users con .info()

In [ ]:
usage.info() # inspección de usage con .info()

---

## 🧩Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**  
- Cuenta valores nulos por columna para cada dataset.
- Calcula la proporción de nulos por columna para cada dataset.

El dataset `plans` solamente tiene 2 renglones y se puede observar que no tiene ausentes, por ello no necesita exploración adicional.

<br>
<details>
<summary>Haz clic para ver la pista</summary>
Usa `.isna().sum()` para contar valores nulos y usa `.isna().mean()` para calcular la proporción.

In [ ]:
# Cantidad de valores nulos para users
print('Cantidad de valores nulos en users:')
print(users.isna().sum())

# Proporción de valores nulos para users
print('\nProporción de valores nulos en users:')
print(users.isna().mean()*100)

In [ ]:
# cantidad de nulos para usage
print('Cantidad de valores nulos en usage:')
print(usage.isna().sum())

# proporción de valores nulos para usage
print('\nProporción de valores nulos en usage:')
print(usage.isna().mean()*100)

✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico al final del bloque. Incluye qué ves y que acción recomendarías para cada caso.

💡 **Nota:** Justifica tus decisiones brevemente (1 línea por caso).
* Hint:
 - Si una columna tiene **más del 80–90% de nulos**, normalmente se **ignora o elimina**.  
 - Si tiene **entre 5% y 30%**, generalmente se **investiga para imputar o dejar como nulos**.  
 - Si es **menor al 5%**, suele ser un caso simple de imputación o dejar como nulos.

 ---

**Valores nulos**  
- ¿Qué columnas tienen valores faltantes y en qué proporción?  
- Indica qué harías: ¿imputar, eliminar, ignorar?

### Diagnóstico de valores nulos

* **`users` — `city` (469 nulos | 11.725%):** Investigar y evaluar imputar (por ejemplo, con `'Desconocido'` o según patrón) o dejar como nulo; está en el rango del 5–30% y conviene revisar si falta de forma aleatoria o por segmento antes de rellenar.
* **`users` — `churn_date` (3,534 nulos | 88.350%):** Dejar como nulo / no imputar con fechas ficticias (y considerar su alcance analítico); supera el 80%, ya que en clientes activos o sin baja registrada es esperable que no exista fecha de cancelación.
* **`usage` — `date` (50 nulos | 0.125%):** Dejar como nulo o evaluar eliminar/imputar puntualmente; es una proporción mínima (<5%) con bajo impacto global sobre el volumen de eventos.
* **`usage` — `duration` (22,076 nulos | 55.190%):** Investigar antes de imputar o eliminar; aunque supera el 30% y está entre los tramos intermedios, en datos de uso suele corresponder a eventos que no son llamadas (como mensajes) o registros sin duración válida.
* **`usage` — `length` (17,896 nulos | 44.740%):** Investigar antes de imputar o eliminar; al estar en un porcentaje intermedio alto, conviene validar si se debe a eventos que no son mensajes (como llamadas) o a datos faltantes reales.



### 2.2 Detección de valores inválidos y sentinels

🎯 **Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numéricas con **un resumen estadístico** y describe brevemente que encontraste.
- Explora las columnas categóricas **relevantes**, revisando sus valores únicos y describe brevemente que encontraste.


El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

In [ ]:
# Explorar columnas numéricas de users
display(users.describe().round(2))

- La columna `user_id` tiene 4,000 registros válidos, va de `10000` a `13999` con una media y mediana idénticas de `11999.50` y desviación estándar de `1154.84`, lo cual es consistente con un identificador secuencial sin valores atípicos ni errores.
- La columna `age` tiene 4,000 registros válidos, pero presenta un valor mínimo de `-999.00` (un *sentinel value* o código de dato faltante/inválido), mientras que el resto de su distribución se ve razonable (mediana de `47.00`, IQR de `32.00` a `63.00` y máximo de `79.00`); ese `-999` distorsiona la media (`33.74`) y eleva la desviación estándar (`123.23`), por lo que conviene convertirlo a `NaN` en la etapa de limpieza.

In [ ]:
# Explorar columnas numéricas de usage
display(usage.describe().round(2))


- Las columnas `id` y `user_id` tienen 40,000 registros válidos cada una: `id` va de `1.00` a `40000.00` (media y mediana en `20000.50`, como corresponde a un identificador secuencial de eventos), mientras que `user_id` oscila entre `10000.00` y `13999.00` (coherente con los 4,000 usuarios de `users`, con media de `12002.41` y mediana de `12013.00`, reflejando distinta frecuencia de actividad por cliente).
- En `duration` (17,924 registros no nulos) los valores van de `0.00` a `120.00` (probablemente en minutos o segundos, con mediana de `3.50`, media de `5.20` y Q3 en `6.99`), lo que muestra una distribución sesgada a la derecha con llamadas muy cortas o en `0.00` y picos altos hasta `120.00`; en `length` (22,104 registros no nulos) los valores van de `0.00` a `1490.00` (mediana de `50.00`, media de `52.13` y Q3 en `64.00`), con una concentración moderada en longitudes cortas y un máximo extremo de `1490.00` que conviene revisar como posible *outlier* o caso atípico.

In [ ]:
# Explorar columnas categóricas de users
# Explorar columnas categóricas de users
columnas_user = ['city', 'plan']


for col in columnas_user:
  print(f'=== {col.upper()} ===')
  print('Valores únicos (incluyendo NaN):', users[col].unique())
  print('Cantidad de categorías únicas (sin NaN):', users[col].nunique())
  print('\nFrecuencia de categorías:')
  print(users[col].value_counts(dropna=False))
  print('\n' + '-' * 40 + '\n')


In [ ]:
# Explorar columnas categóricas de users
columnas_user = ['city', 'plan']

for col in columnas_user:
  print(f'--- {col} ---')
  print(users[col].value_counts(dropna=False))
  print()

- En la columna `city` aparecen 8 categorías distintas (incluyendo nulos y un marcador de faltante): las ciudades con mayor presencia son **Bogotá** (808), **CDMX** (730) y **Medellín** (616), seguidas por **GDL** (450), **Cali** (424) y **MTY** (407); además, se detectan **469 valores nulos (`NaN`)** y **96 registros con `'?'`** (un valor centinela que representa dato faltante/desconocido y que en la limpieza conviene unificar con `NaN` o `'Desconocido'`, y también estandarizar nomenclaturas si se requiere agrupar por país o ciudad).
- En la columna `plan` hay 2 categorías bien definidas y sin valores nulos: **Basico** predomina con **2,595 usuarios** (~64.9% de la base) frente a **1,405 usuarios** en **Premium** (~35.1%), lo que muestra una mayoría de clientes en el plan de entrada y una base relevante para analizar oportunidades de *upgrade* o retención.

In [ ]:

# Explorar columna categórica de usage
print('Valores únicos (incluyendo NaN):', usage['type'].unique())
print('Cantidad de categorías únicas (sin NaN):', usage['type'].nunique())
print('\nFrecuencia absoluta:')
print(usage['type'].value_counts(dropna=False))
print('\nFrecuencia relativa:')
print(usage['type'].value_counts(normalize=True, dropna=False).round(4))

In [ ]:
usage['type'].value_counts(dropna=False)

- La columna `type` contiene dos categorías válidas y sin valores nulos: **`text`** predomina con **22,092 registros** (~55.2% del total de eventos) frente a **`call`** con **17,908 registros** (~44.8%).

---
✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico. Incluye qué ves y que acción recomendarías para cada caso.

**Valores inválidos o sentinels**  
- ¿En qué columnas encontraste valores inválidos o sentinels?  
- ¿Qué acción tomarías?

Rta/ ### Diagnóstico de valores inválidos o sentinels

- **`users` — `age`:** Se detecta el valor centinela `-999` (edad imposible que distorsiona la media y la desviación estándar). *Acción:* Reemplazar `-999` por `NaN` (por ejemplo, con `.replace(-999, pd.NA)` o `np.nan`) para tratarlo como dato faltante real y evitar sesgos en análisis demográficos.
- **`users` — `city`:** Aparece el valor centinela `'?'` (96 registros) como marcador de ciudad desconocida, además de nulos (`NaN`). *Acción:* Reemplazar `'?'` por `NaN` (o unificar con `'Desconocido'` según el criterio del análisis) y verificar si conviene estandarizar formatos de texto.
- **`usage` — `duration` y `length`:** Se observan valores mínimos de `0.00` y máximos altos (`120.00` en `duration` y `1490.00` en `length`). *Acción:* No eliminar a ciegas sin validar; revisar el contexto por tipo de evento (`call` vs. `text`) y analizar si `0.00` o los valores altos corresponden a eventos válidos (llamadas cortas/sin conexión o mensajes largos) o a registros atípicos/errores.

### 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**  
- Convierte las columnas de fecha a tipo fecha y asegurate de que el código sea a prueba de errores.  
- Revisa cuántas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Toma en cuenta que tenemos datos registrados hasta el año 2024.

In [ ]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')

In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'], errors='coerce')

In [ ]:
# Revisar los años presentes en `reg_date` de users
print('Frecuencia por año en users["reg_date"]:')
print(users['reg_date'].dt.year.value_counts(dropna=False).sort_index())

En `reg_date` podemos observar que los datos entre el 2022 y 2024 se concentran de manera equilibrada (las mismas proporciones), para el año 2026 se presentan 40 datos, esto es una anomalia en la base de datos ya que se estan evaluando datos hasta el 2024

In [ ]:
# Revisar los años presentes en `date` de usage
print('\nFrecuencia por año en usage["date"]:')
print(usage['date'].dt.year.value_counts(dropna=False).sort_index())

La inmensa mayoría de los registros (39,950 eventos) corresponden exclusivamente al año 2024, dentro del rango válido esperado del proyecto.
Ademas podemos observar que existen 50 registros sin fecha (NaN), sin años futuros ni fechas imposibles detectadas en esta columna.

✍️ **Comentario**: escribe tu diagnóstico, e incluye **qué acción recomendarías** para cada caso:

**Fechas fuera de rango**  
- ¿Aparecen años imposibles? (años muy viejos o sin transcurrir al momento de guardar los datos)
- ¿Qué harías con ellas?

Rta/ ### Diagnóstico de fechas fuera de rango

- **`users` — `reg_date`:** Sí aparecen años futuros / fuera de rango respecto al corte de datos hasta 2024: se identifican. *Acción:* Inspeccionar esos 40 registros de `2026` y, dado que el dataset tiene registros hasta 2024, marcarlos/filtrarlos o excluirlos del análisis temporal válido (o convertir a `NaT` si corresponden a errores de captura/carga) para no sesgar antigüedad ni cohortes de registro.
- **`usage` — `date`:** No aparecen años futuros ni años imposibles; con año válido corresponden exclusivamente a 2024, y solo se observan 50 valores nulos (`NaT`). *Acción:* Mantener los registros válidos de `2024` como base del análisis de uso y dejar los 50 `NaT` sin inventar fechas (o excluirlos únicamente en cálculos que exijan fecha exacta.

---

## 🧩Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**  
- En `age`, reemplaza el sentinel **-999** con la mediana.
- En `city`, reemplaza el sentinel `"?"` por valores nulos (`pd.NA`).  
- Marca como nulas (`pd.NA`) las fechas fuera de rango.

In [ ]:
# Reemplazar -999 por la mediana de age
age_mediana = users.loc[users['age'] != -999, 'age'].median()
users['age'] = users['age'].replace(-999, age_mediana)

# Verificar cambios
display(users['age'].describe())

In [ ]:
# Reemplazar ? por NA en city
users['city'] = users['city'].replace('?', pd.NA)

# Verificar cambios
print(users['city'].value_counts(dropna=False))


In [ ]:

# Marcar fechas futuras como NA para reg_date
users.loc[users['reg_date'].dt.year > 2024, 'reg_date'] = pd.NaT

# Verificar cambios
print(users['reg_date'].dt.year.value_counts(dropna=False).sort_index())


### 3.2 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Decidir qué hacer con los valores nulos según su proporción y relevancia.

**Instrucciones:**
- Verifica si los nulos en `duration` y `length` son **MAR**(Missing At Random) revisando si dependen de la columna `type`.  
  Si confirmas que son MAR, **déjalos como nulos** y justifica la decisión.

In [ ]:
# Verificación MAR en usage (Missing At Random) para duration
print("Proporción de nulos en 'duration' según 'type':")
print(usage['duration'].isna().groupby(usage['type']).mean())



In [ ]:
# Verificación MAR en usage (Missing At Random) para length
print("\nProporción de nulos en 'length' según 'type':")
print(usage['length'].isna().groupby(usage['type']).mean())

Haz doble clic aquíy escribe que tu diagnostico de nulos en `duration` y `length`

- **Diagnóstico MAR:** Los valores faltantes en `duration` y `length` son **MAR (*Missing At Random*)**, ya que dependen de la columna `type` en `duration` hay **0% de nulos en `call`** y **~99.93% en `text`**, mientras que en `length` hay **~99.93% de nulos en `call`** y **0% en `text`**
- **Decisión y justificación:** Se **dejan como nulos (`NaN`)** sin imputar. Cada métrica aplica a un tipo de evento distinto (duración para llamadas y longitud para mensajes de texto); si se rellena con `0` o con un promedio/mediana se introduciría ruido y sesgaría los análisis por servicio.

---

## 🧩Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso

🎯**Objetivo**: Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico.

**Instrucciones:**:
1. Construye una tabla agregada de `usage` por `user_id` que incluya:
- número total de mensajes  
- número total de llamadas  
- total de minutos de llamadas

2. Renombra las columnas para que tengan nombres claros:  
- `cant_mensajes`  
- `cant_llamadas`  
- `cant_minutos_llamada`
3. Combina esta tabla con `users`.

In [ ]:
# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int)  # conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int)  # conocer el total de llamadas

# Agrupar información por usuario y renombrar columnas
usage_agg = (
    usage.groupby("user_id")
    .agg(
        cant_mensajes=("is_text", "sum"),
        cant_llamadas=("is_call", "sum"),
        cant_minutos_llamada=("duration", "sum"),
    )
    .reset_index()
)

# Observar resultado
usage_agg.head(3)

In [ ]:
# Renombrar columnas
usage_agg = usage_agg.rename(
    columns={
        "is_text": "cant_mensajes",
        "is_call": "cant_llamadas",
        "duration": "cant_minutos_llamada",
    }
)

# Observar resultado
usage_agg.head(3)

In [ ]:
# Combinar la tabla agregada con el dataset de usuarios
user_profile = pd.merge(users, usage_agg, on="user_id", how="left")

# Mostrar las primeras 5 filas
user_profile.head(5)

### 4.2 4.2 Resumen estadístico por usuario durante el 2024

🎯 **Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios, para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

**Instrucciones:**  
1. Para las columnas **numéricas** relevantes, obtén un resumen estadístico (media, mediana, mínimo, máximo, etc.).  
2. Para la columna **categórica** `plan`, revisa la distribución en **porcentajes** de cada categoría.

In [ ]:
# Resumen estadístico de las columnas numéricas
cols_numericas = [
    col
    for col in ["age", "cant_mensajes", "cant_llamadas", "cant_minutos_llamada"]
    if col in user_profile.columns
]

print("Resumen estadístico de columnas numéricas (user_profile):")
display(user_profile[cols_numericas].describe())

In [ ]:
# Distribución porcentual del tipo de plan
print("\nDistribución porcentual de `plan`:")
print((user_profile["plan"].value_counts(dropna=False, normalize=True) * 100).round(2))

---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones

🎯 **Objetivo:**  
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

**Instrucciones:**  
Graficar **histogramas** para las siguientes columnas:  
- `age` (edad de los usuarios)
- `cant_mensajes`
- `cant_llamadas`
- `total_minutos_llamada`

Después de cada gráfico, escribe un **insight** respecto al plan y la variable, por ejemplo:  
- "Dentro del plan Premium, hay mayor proporción de..."  
- "Los usuarios Básico tienden a hacer ... llamadas y enviar ... mensajes."  o "No existe algún patrón."
- ¿Qué tipo de distribución tiene ? (simétrica, sesgada a la derecha o a la izquierda)

**Hint**  
Para cada histograma,
- Usa `hue='plan'` para ver cómo varían las distribuciones según el plan (Básico o Premium).
- Usa `palette=['skyblue','green']`
- Agrega título y etiquetas

In [ ]:

# Histograma para visualizar la edad (age)
plt.figure(figsize=(8, 4.5))
sns.histplot(
    data=user_profile,
    x="age",
    hue="plan",
    palette=["skyblue", "green"],
    kde=True,
    alpha=0.5,
    bins=20,
)
plt.title("Distribución de la edad de los usuarios según plan")
plt.xlabel("Edad (años)")
plt.ylabel("Cantidad de usuarios")
plt.show()

💡Insights:

 - Distribución amplia y relativamente uniforme (sin sesgo pronunciado) entre los ~18 y ~80 años, con mayor volumen de usuarios en el plan **Básico** frente al **Premium** a lo largo de la mayoría de las edades, sin un patrón etario marcadamente exclusivo de un plan.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograma para visualizar la cant_mensajes
plt.figure(figsize=(8, 4.5))
sns.histplot(
    data=user_profile,
    x="cant_mensajes",
    hue="plan",
    palette=["skyblue", "green"],
    kde=True,
    alpha=0.5,
    bins=25,
)
plt.title("Distribución de cantidad de mensajes según plan")
plt.xlabel("Cantidad de mensajes")
plt.ylabel("Frecuencia de usuarios")
plt.show()


💡Insights:
- - Distribución concentrada en torno a los 4–6 mensajes (forma acampanada/simétrica con ligera cola a la derecha), donde el plan **Básico** supera en frecuencia al **Premium** en casi todos los tramos, pero ambos mantienen un comportamiento de mensajería bastante similar.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograma para visualizar la cant_llamadas
plt.figure(figsize=(8, 4.5))
sns.histplot(
    data=user_profile,
    x="cant_llamadas",
    hue="plan",
    palette=["skyblue", "green"],
    kde=True,
    alpha=0.5,
    bins=25,
)
plt.title("Distribución de cantidad de llamadas según plan")
plt.xlabel("Cantidad de llamadas")
plt.ylabel("Frecuencia de usuarios")
plt.show()


💡Insights:
- Distribución concentrada principalmente entre 2 y 7 llamadas (pico alrededor de 3–4 y ligera cola a la derecha), donde el plan **Basico** tiene mayor frecuencia que el **Premium** en casi todos los tramos, pero ambos presentan un comportamiento muy similar en la cantidad de llamadas.

In [ ]:
# Histograma para visualizar la cant_minutos_llamada
plt.figure(figsize=(8, 4.5))
sns.histplot(
    data=user_profile,
    x="cant_minutos_llamada",
    hue="plan",
    palette=["skyblue", "green"],
    kde=True,
    alpha=0.5,
    bins=25,
)
plt.title("Distribución de minutos de llamada según plan")
plt.xlabel("Cantidad de minutos de llamada")
plt.ylabel("Frecuencia de usuarios")
plt.show()


💡Insights:
- Distribución sesgada a la derecha (*right-skewed*), concentrada principalmente entre 0 y 45 minutos (pico alrededor de 15–20 min) con cola hacia la derecha y algunos valores altos aislados cerca de 140–160 min; el plan **Basico** tiene mayor frecuencia que el **Premium** en casi todo el rango, pero ambos mantienen un comportamiento similar.

### 5.2 Identificación de Outliers

🎯 **Objetivo:**  
Detectar valores extremos en las variables clave de **uso** y **clientes** que podrían afectar el análisis, y decidir si requieren limpieza o revisión adicional.

**Instrucciones:**  
- Usa **boxplots** para identificar visualmente outliers en las siguientes columnas:  
  - `age`
  - `cant_mensajes`
  - `cant_llamadas`
  - `total_minutos_llamada`  
- Crea un **for** para generar los 4 boxplots automáticamente.
<br>

- Después de crear los gráfico, responde si **existen o no outliers** en las variables.  
- Si hay outliers, crea otro bucle para calcular los límites de esas columnas usando el **método IQR** y decide qué hacer con ellos.
  - Si solamente hay outliers de un solo lado, no es necesario calcular ambos límites.

**Hint:**
- Dentro del bucle, usa `plt.title(f'Boxplot: {col}')` para que el título cambie acorde a la columna.

In [ ]:
# Visualizando usando BoxPlot
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    plt.figure(figsize=(8, 3.5))
    sns.boxplot(x=user_profile[col], color='skyblue')
    plt.title(f'Boxplot: {col}')
    plt.xlabel(col)
    plt.show()

💡Insights:
- Age: La mediana en la grafica de Age se ubica cerca de los 48 años, con el rango intercuartílico (IQR) aproximadamente entre 33 y 63 años, los bigotes cubren todo el rango desde ~18 hasta ~79 años y no hay outliers (la distribución es amplia y bastante equilibrada sin puntos atípicos fuera de los bigotes)
- cant_mensajes: La mediana de la grafica Cant-mensajes está alrededor de los 5 mensajes, el IQR se concentra entre ~4 y 7 mensajes, el bigote superior llega cerca de 11 y sí existen outliers en el extremo superior (puntos aislados desde 12 hasta 17 mensajes), lo cual coincide con la ligera cola hacia la derecha vista antes en el histograma.
- cant_llamadas: Presenta outliers superiores (entre ~11 y 15 llamadas), con mediana cercana a 4 e IQR entre ~3 y 6 llamadas, confirmando la cola derecha moderada.
- cant_minutos_llamada: Presenta una cantidad notable de outliers superiores (desde ~63 hasta ~155–160 minutos), con mediana cerca de 20 minutos e IQR entre ~11 y 31 minutos, coherente con el sesgo a la derecha observado en el histograma....

In [ ]:
# Calcular límites con el método IQR
columnas_limites = ["age", "cant_mensajes", "cant_llamadas", "cant_minutos_llamada"]

for col in columnas_limites:
    q1 = user_profile[col].quantile(0.25)
    q3 = user_profile[col].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = user_profile[
        (user_profile[col] < limite_inferior) | (user_profile[col] > limite_superior)
    ]
    pct_outliers = (len(outliers) / len(user_profile)) * 100

    print(f"=== {col} ===")
    print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
    print(f"Límite inferior: {limite_inferior:.2f} | Límite superior: {limite_superior:.2f}")
    print(f"Cantidad de outliers: {len(outliers)} ({pct_outliers:.2f}%)\n")


In [ ]:

# Revisa los limites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()


💡Insights:
- cant_mensajes: Mantener outliers; mediana en 5.0, Q3 en 7.0 y máx. en 17.0. Representan usuarios con mayor envío de mensajes (cola derecha válida), no errores, por lo que conservarlos mantiene la representatividad y conviene usar la mediana como referencia.
- cant_llamadas: Mantener outliers; mediana en 4.0, Q3 en 6.0 y máx. en 15.0. Reflejan clientes con mayor frecuencia de llamadas dentro de un rango real de uso; eliminarlos sesgaría el perfil de actividad.
- cant_minutos_llamada: Mantener outliers; mediana en 19.78, Q3 en 31.42 y máx. en 155.69. Indican usuarios con mayor consumo acumulado de minutos (sesgo a la derecha); se conservan por ser comportamiento real y se priorizan métricas robustas (mediana/IQR) sobre la media.

---

## 🧩Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso

🎯 **Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_uso` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones de llamadas y mensajes y asigna:
  - `'Bajo uso'` cuando llamadas < 5 y mensajes < 5
  - `'Uso medio'` cuando llamadas < 10 y mensajes < 10
  - `'Alto uso'` para el resto de casos

In [ ]:
# Crear columna grupo_uso
def clasificar_uso(row):
    llamadas = row["cant_llamadas"]
    mensajes = row["cant_mensajes"]

    # Manejo seguro si algún usuario no tiene registros de uso (NaN)
    if pd.isna(llamadas) or pd.isna(mensajes):
        return "Sin datos"

    if llamadas < 5 and mensajes < 5:
        return "Bajo uso"
    elif llamadas < 10 and mensajes < 10:
        return "Uso medio"
    else:
        return "Alto uso"

user_profile["grupo_uso"] = user_profile.apply(clasificar_uso, axis=1)

# Verificar distribución de segmentos
print(user_profile["grupo_uso"].value_counts(dropna=False))

In [ ]:
# verificar cambios
user_profile.head()

### 6.2 Segmentación de Clientes Por Edad

🎯 **Objetivo:**: Clasificar a cada usuario en un grupo por **edad**.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_edad` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones y asigna:
  - `'Joven'` cuando age < 30
  - `'Adulto'` cuando age < 60
  - `'Adulto Mayor'` para el resto de casos

In [ ]:
# Crear columna grupo_edad
def clasificar_edad(edad):
  if pd.isna(edad):
    return "Sin datos"
  if edad < 30:
    return "Joven"
  elif edad < 60:
    return "Adulto"
  else:
    return "Adulto Mayor"


user_profile["grupo_edad"] = user_profile["age"].apply(clasificar_edad)

# Verificar distribución por grupo de edad
print(user_profile["grupo_edad"].value_counts(dropna=False))

In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la Segmentación de Clientes

🎯 **Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

**Instrucciones:**  
- Crea dos gráficos para las variables categóricas `grupo_uso` y `grupo_edad`.
- Agrega título y etiquetas a los ejes en cada gráfico.

In [ ]:
# Visualización de los segmentos por uso
sns.countplot(
    data=user_profile,
    x="grupo_uso",
    order=["Bajo uso", "Uso medio", "Alto uso", "Sin datos"],
    palette="Blues_d",
)
plt.title("Distribución de usuarios según grupo de uso")
plt.xlabel("Grupo de uso")
plt.ylabel("Cantidad de usuarios")
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(
    data=user_profile,
    x="grupo_edad",
    order=["Joven", "Adulto", "Adulto Mayor", "Sin datos"],
    palette="Greens_d",
)
plt.title("Distribución de usuarios según grupo de edad")
plt.xlabel("Grupo de edad")
plt.ylabel("Cantidad de usuarios")
plt.show()


---
## 🧩Paso 7: Insight Ejecutivo para Stakeholders

🎯 **Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

**Preguntas a responder:**
- ¿Qué problemas tenían originalmemte los datos?¿Qué porcentaje, o cantidad de filas, de esa columna representaban?


- ¿Qué segmentos de clientes identificaste y cómo se comportan según su edad y nivel de uso?  
- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?  
- ¿Qué patrones de uso extremo (outliers) encontraste y qué implican para el negocio?


- ¿Qué recomendaciones harías para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?

✍️ **Escribe aquí tu análisis ejecutivo:**

### Análisis ejecutivo

⚠️ **Problemas detectados en los datos**
- abc
- abc


🔍 **Segmentos por Edad**

- **Criterio de segmentación:** Se clasificó a los 4,000 usuarios (rango de 18 a 79 años, mediana de 48 años e IQR de 33 a 63 años) en tres grupos etarios: **Joven** (`< 30` años), **Adulto** (`30–59` años) y **Adulto Mayor** (`≥ 60` años).
- **Distribución y peso en la base:** El segmento **Adulto** concentra la mayor proporción de clientes al abarcar el rango etario más amplio, seguido de **Adulto Mayor** y **Joven**, mostrando una base demográfica equilibrada y sin valores atípicos (*outliers*) en la edad.
- **Comportamiento de uso:** La edad no genera distorsiones extremas por sí sola, por lo que dentro de cada rango etario conviven distintos niveles de consumo; sin embargo, el segmento **Adulto** representa el núcleo principal de volumen de usuarios para el negocio, mientras que **Joven** y **Adulto Mayor** ofrecen oportunidades diferenciadas de comunicación y diseño de planes según sus hábitos de llamada y mensajería.
-
📊 **Segmentos por Nivel de Uso**
* **Criterio de segmentación:** Se clasificó a los usuarios según la combinación de `cant_llamadas` y `cant_mensajes` en tres grupos: **Bajo uso** (`< 5` llamadas y `< 5` mensajes), **Uso medio** (`< 10` llamadas y `< 10` mensajes) y **Alto uso** (el resto de casos, incluyendo quienes alcanzan o superan los umbrales de 10 o más en llamadas o mensajes, o combinaciones superiores).
* **Distribución y comportamiento:** La mayor parte de los clientes se concentra en niveles bajos a medios de actividad (en línea con las medianas cercanas a 4 llamadas, 5 mensajes y 19.78 minutos de llamada), mientras que **Alto uso** representa un segmento más reducido pero de mayor intensidad en la cola derecha, con patrones similares entre los planes **Básico** y **Premium** y un mayor volumen general de usuarios en el plan **Básico**.
* **Implicación comercial:** **Bajo uso** requiere estrategias de activación y retención de bajo costo; **Uso medio** ofrece margen para incentivar mayor recurrencia y acercarlo a planes superiores; y **Alto uso** es clave para cuidar la satisfacción, prevenir fricciones o fugas y proponer ofertas de *upselling* alineadas a su alto consumo.

➡️ Esto sugiere que el negocio no enfrenta un problema de datos corruptos en los valores extremos, sino una distribución asimétrica donde conviven clientes de actividad moderada con un grupo minoritario de alto consumo. Al mantener los outliers válidos y analizar los usuarios según su nivel de uso (Bajo uso, Uso medio, Alto uso), su franja etaria (Joven, Adulto, Adulto Mayor) y su plan actual (Básico vs. Premium), se abre la puerta a decisiones comerciales más precisas que tratar a toda la base como homogénea.


💡 **Recomendaciones**
Se recomiendan paquetes adicionales para usuarios de alto uso, campañas de activación para usuarios de uso medio y planes claros para usuarios de bajo uso. Se sugiere completar la cobertura de datos y monitorear cambios en el uso para detectar oportunidades de upselling y churn.

---

## 🧩Paso 8 Cargar tu notebook y README a GitHub

🎯 **Objetivo:**  
Entregar tu análisis de forma **profesional**, **documentada** y **versionada**, asegurando que cualquier persona pueda revisar, ejecutar y entender tu trabajo.



### Opción A : Subir archivos desde la interfaz de GitHub (UI)

1. Descarga este notebook (`File → Download .ipynb`).  
2. Entra a tu repositorio en GitHub (por ejemplo `telecom-analysis` o `sprint7-final-project`).  
3. Sube tu notebook **Add file → Upload files**.  

---

### Opción B : Guardar directo desde Google Colab

1. Abre tu notebook en Colab.  
2. Ve a **File → Save a copy in GitHub**.  
3. Selecciona el repositorio y la carpeta correcta (ej: `notebooks/`).  
4. Escribe un mensaje de commit claro, por ejemplo:  
    - `feat: add final ConnectaTel analysis`
    - `agregar version final: Análisis ConnectaTel`
5. Verifica en GitHub que el archivo quedó en el lugar correcto y que el historial de commits se mantenga limpio.

---

Agrega un archivo `README.md` que describa de forma clara:
- el objetivo del proyecto,  
- los datasets utilizados,  
- las etapas del análisis realizadas,  
- cómo ejecutar el notebook (por ejemplo, abrirlo en Google Colab),  
- una breve guía de reproducción.
---

Link a repositorio público del proyecto: `LINK a tu repo aquí`